In [4]:
import torch
import torchvision.transforms as transforms
from torch.utils.data import Dataset, DataLoader
from PIL import Image
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

In [5]:
metadata_dir = '/export/usuarios_ml4ds/danibacaicoa/ForwardBackard_losses_old/Datasets/raw_datasets/Clothing1M/'
clean_label_kv_path = os.path.join(metadata_dir, 'clean_label_kv.txt')
noisy_label_kv_path = os.path.join(metadata_dir, 'noisy_label_kv.txt')
clean_train_key_list_path = os.path.join(metadata_dir, 'clean_train_key_list.txt')
noisy_train_key_list_path = os.path.join(metadata_dir, 'noisy_train_key_list.txt')
clean_val_key_list_path = os.path.join(metadata_dir, 'clean_val_key_list.txt')
clean_test_key_list_path = os.path.join(metadata_dir, 'clean_test_key_list.txt')
category_names_eng_path = os.path.join(metadata_dir, 'category_names_eng.txt')


In [6]:
def load_labels(filepath):
    '''
    loads a dict {path_1:label_1,path_2:label_2} for a given file.
    '''
    labels = {}
    with open(filepath, 'r') as f:
        for line in f:
            parts = line.strip().split()
            if len(parts) == 2:
                image_path = os.path.normpath(parts[0])
                labels[image_path] = int(parts[1])
            else:
                pass

    return labels

def load_key_list(filepath):
    keys = []
    with open(filepath, 'r') as f:
        for line in f:
            image_path = os.path.normpath(line.strip())
            keys.append(image_path)
    return keys

In [7]:
# Load category names
category_names = None
with open(category_names_eng_path, 'r') as f:
    category_names = [line.strip() for line in f if line.strip()]

print(f"Loaded {len(category_names)} category names.")
print(category_names)

Loaded 14 category names.
['T-Shirt', 'Shirt', 'Knitwear', 'Chiffon', 'Sweater', 'Hoodie', 'Windbreaker', 'Jacket', 'Downcoat', 'Suit', 'Shawl', 'Dress', 'Vest', 'Underwear']


In [12]:
class Clothing1MDataset(Dataset):
    def __init__(self, root_dir, split='train', transform=None):
        self.root_dir = root_dir
        self.transform = transform
        self.split = split
        self.samples = [] 

        clean_labels = load_labels(clean_label_kv_path)
        noisy_labels = load_labels(noisy_label_kv_path)

        

        self.N = np.zeros((len(category_names), len(category_names)), dtype=int)
        print(f"Loaded {len(clean_labels)} clean labels and {len(noisy_labels)} noisy labels.")


        if split == 'train':
            clean_keys = load_key_list(clean_train_key_list_path)
            noisy_keys = load_key_list(noisy_train_key_list_path)


            for key in noisy_keys:
                self.samples.append((key, noisy_labels[key], 1))
                if key in clean_keys:
                    self.N[noisy_labels[key], clean_labels[key]] += 1
            for key in clean_keys:
                if key not in noisy_keys:
                    self.samples.append((key, clean_labels[key], 0))

        elif split == 'test':
            test_keys = load_key_list(clean_test_key_list_path)
            for key in test_keys:
                self.samples.append((key, clean_labels[key], 0))



    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        if torch.is_tensor(idx):
            idx = idx.tolist()

        relative_img_path, label, noisy_flag = self.samples[idx]
        img_full_path = os.path.join(self.root_dir, relative_img_path)


        image = Image.open(img_full_path).convert('RGB')
        image = self.transform(image)


        return image, label, noisy_flag

In [15]:
class Clothing1MDataset(Dataset):
    def __init__(self, root_dir, split='train', transform=None):
        self.root_dir = root_dir
        self.transform = transform
        self.split = split
        self.samples = []

        print(f"Loading labels for split '{split}'...") # Initial message
        # It's often efficient to load all labels upfront if memory allows
        clean_labels = load_labels(clean_label_kv_path)
        noisy_labels = load_labels(noisy_label_kv_path)
        print(f"Loaded {len(clean_labels)} clean labels and {len(noisy_labels)} noisy labels.")

        # Assuming category_names is loaded globally or passed in
        global category_names
        self.N = np.zeros((len(category_names), len(category_names)), dtype=int)

        processed_count = 0
        # Print update every N images to avoid excessive I/O
        print_interval = 10000 # Adjust as needed

        if split == 'train':
            print("Processing training keys...")
            clean_keys = load_key_list(clean_train_key_list_path)
            noisy_keys = load_key_list(noisy_train_key_list_path)
            total_keys_approx = len(noisy_keys) + len(clean_keys) # Rough estimate

            # --- Process noisy keys ---
            noisy_keys_set = set(noisy_keys) # Convert to set for efficient lookup later
            for key in noisy_keys:
                if key in noisy_labels: # Ensure label exists for this key
                    self.samples.append((key, noisy_labels[key], 1)) # 1 = noisy flag
                    processed_count += 1
                    # Update noise matrix only if clean label also known
                    if key in clean_labels: # Check if this key has a known clean label
                        self.N[noisy_labels[key], clean_labels[key]] += 1

                    # Print progress periodically
                    if processed_count % print_interval == 0:
                        print(f"  Processed ~{processed_count}/{total_keys_approx} training samples...", end='\r')
                else:
                    # Optional: Log warning for missing labels
                    # print(f"\nWarning: Noisy key {key} missing from noisy_labels.txt", end='')
                    pass

            # --- Process clean keys that are NOT in the noisy set ---
            clean_keys_set = set(clean_keys) # Use set for faster lookup
            for key in clean_keys_set:
                if key not in noisy_keys_set: # Only add clean samples not already added from noisy list
                     if key in clean_labels: # Ensure label exists for this key
                        self.samples.append((key, clean_labels[key], 0)) # 0 = clean flag
                        processed_count += 1
                        # Print progress periodically
                        if processed_count % print_interval == 0:
                             print(f"  Processed ~{processed_count}/{total_keys_approx} training samples...", end='\r')
                     else:
                        # Optional: Log warning for missing labels
                        # print(f"\nWarning: Clean key {key} missing from clean_labels.txt", end='')
                        pass

            # Final print for training split after loop finishes
            # Overwrite the last progress message and add a newline
            print(f"  Processed {processed_count} total training samples.              ")

        elif split == 'test':
            # --- Process test keys (assuming fix is applied) ---
            print("Processing test keys...")
            test_keys = load_key_list(clean_test_key_list_path)
            total_keys = len(test_keys)
            for i, key in enumerate(test_keys):
                 if key in clean_labels: # Ensure label exists
                     self.samples.append((key, clean_labels[key], 0)) # Test is clean, flag 0
                     processed_count += 1
                     # Print progress periodically or on the last item
                     if processed_count % print_interval == 0 or i == total_keys - 1:
                         print(f"  Processed {processed_count}/{total_keys} test samples...", end='\r')
                 else:
                     # Optional: Log warning for missing labels
                     # print(f"\nWarning: Test key {key} missing from clean_labels.txt", end='')
                     pass
             # Final print for test split
            print(f"  Processed {processed_count} total test samples.              ")

        # Add similar logic for split == 'val' if you implement it

        print(f"Finished loading {len(self.samples)} samples for split '{split}'.") # Final confirmation


    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        if torch.is_tensor(idx):
            idx = idx.tolist()

        relative_img_path, label, noisy_flag = self.samples[idx]
        img_full_path = os.path.join(self.root_dir, relative_img_path)

        try:
            image = Image.open(img_full_path).convert('RGB')
            if self.transform:
                image = self.transform(image)
            return image, label, noisy_flag
        except FileNotFoundError:
            print(f"\nError: Image file not found at {img_full_path}")
            # Handle error appropriately: maybe return None or a placeholder?
            # Or raise an exception if you want DataLoader to skip it (might need custom collate_fn)
            # For simplicity, returning None might cause issues later in training loop
            # A better approach might be to filter out missing files during __init__
            # or raise the error. Let's raise it here for clarity.
            raise FileNotFoundError(f"Image file not found: {img_full_path}")
        except Exception as e:
            print(f"\nError loading image {img_full_path}: {e}")
            raise e # Re-raise other image loading errors



In [16]:
img_size = 224
train_transform = transforms.Compose([
    transforms.Resize((img_size, img_size)),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
    ])
val_test_transform = transforms.Compose([
    transforms.Resize((img_size, img_size)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
    ])

In [17]:
clothing1m_root = '/export/usuarios_ml4ds/danibacaicoa/ForwardBackard_losses_old/Datasets/raw_datasets/Clothing1M/'
train_dataset = None
test_dataset = None
train_loader = None
test_loader = None

train_dataset = Clothing1MDataset(root_dir=clothing1m_root, split='train', transform=train_transform)
print(f"Loaded {len(train_dataset)} training samples.")

test_dataset = Clothing1MDataset(root_dir=clothing1m_root, split='test', transform=val_test_transform)
print(f"Loaded {len(test_dataset)} test samples.")

N = train_dataset.N 

batch_size = 64 
num_workers = 2

train_loader = DataLoader(
    train_dataset,
    batch_size=batch_size,
    shuffle=True,
    num_workers=num_workers,
    pin_memory=True,
    drop_last=True)
test_loader = DataLoader(
    test_dataset,
    batch_size=batch_size * 2,
    shuffle=False,
    num_workers=num_workers,
    pin_memory=True
    )

Loading labels for split 'train'...
Loaded 72409 clean labels and 1037497 noisy labels.
Processing training keys...
  Processed 1047570 total training samples.              
Finished loading 1047570 samples for split 'train'.
Loaded 1047570 training samples.
Loading labels for split 'test'...
Loaded 72409 clean labels and 1037497 noisy labels.
Processing test keys...
  Processed 10526 total test samples.              
Finished loading 10526 samples for split 'test'.
Loaded 10526 test samples.


In [19]:
train_dataset.N

array([[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
       [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
       [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
       [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
       [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
       [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
       [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
       [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
       [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
       [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
       [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
       [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
       [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
       [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]])

In [20]:
print("\n--- First 4 samples from train_dataset ---")
if 'train_dataset' in globals() and train_dataset is not None:
    num_samples_to_show = 4
    if len(train_dataset) < num_samples_to_show:
        print(f"Warning: Train dataset only has {len(train_dataset)} samples.")
        num_samples_to_show = len(train_dataset)

    for i in range(num_samples_to_show):
        try:
            # Get individual sample using index from the Dataset
            image_tensor, label, noisy_flag = train_dataset[i]

            print(f"\nTrain Sample {i}:")
            print(f"  Image Tensor Shape: {image_tensor.shape}") # e.g., torch.Size([3, 224, 224])
            print(f"  Label: {label}")                         # e.g., 5
            print(f"  Is Noisy: {'Yes' if noisy_flag == 1 else 'No'}") # e.g., Yes or No

        except Exception as e:
            print(f"  Error loading sample {i}: {e}")
            break # Stop if error
else:
    print("Train dataset not found or not initialized.")


print("\n\n--- First 4 samples from test_dataset ---")
# Assumes test_dataset is correctly populated (bug fixed in Dataset class)
if 'test_dataset' in globals() and test_dataset is not None and len(test_dataset) > 0:
    num_samples_to_show = 4
    if len(test_dataset) < num_samples_to_show:
        print(f"Warning: Test dataset only has {len(test_dataset)} samples.")
        num_samples_to_show = len(test_dataset)

    for i in range(num_samples_to_show):
        try:
            # Get individual sample using index from the Dataset
            image_tensor, label, noisy_flag = test_dataset[i]

            print(f"\nTest Sample {i}:")
            print(f"  Image Tensor Shape: {image_tensor.shape}") # e.g., torch.Size([3, 224, 224])
            print(f"  Label: {label}")                         # e.g., 10
            print(f"  Is Noisy: {'Yes' if noisy_flag == 1 else 'No'}") # Should always be 'No' for test

        except Exception as e:
            print(f"  Error loading sample {i}: {e}")
            break # Stop if error

elif 'test_dataset' in globals() and test_dataset is not None and len(test_dataset) == 0:
    print("Test dataset found, but it is empty. Check for loading errors or the test split bug.")
else:
     print("Test dataset not found or not initialized.")


--- First 4 samples from train_dataset ---

Train Sample 0:
  Image Tensor Shape: torch.Size([3, 224, 224])
  Label: 0
  Is Noisy: Yes

Train Sample 1:
  Image Tensor Shape: torch.Size([3, 224, 224])
  Label: 0
  Is Noisy: Yes

Train Sample 2:
  Image Tensor Shape: torch.Size([3, 224, 224])
  Label: 0
  Is Noisy: Yes

Train Sample 3:
  Image Tensor Shape: torch.Size([3, 224, 224])
  Label: 0
  Is Noisy: Yes


--- First 4 samples from test_dataset ---

Test Sample 0:
  Image Tensor Shape: torch.Size([3, 224, 224])
  Label: 13
  Is Noisy: No

Test Sample 1:
  Image Tensor Shape: torch.Size([3, 224, 224])
  Label: 11
  Is Noisy: No

Test Sample 2:
  Image Tensor Shape: torch.Size([3, 224, 224])
  Label: 12
  Is Noisy: No

Test Sample 3:
  Image Tensor Shape: torch.Size([3, 224, 224])
  Label: 11
  Is Noisy: No
